# Chapter 28
## Weakly Coupled Oscillators
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter28.ipynb)

## About this chapter

When coupling between two oscillators is weak, the full neuron equations reduce
to slowly changing phases: each cell's basic cycle survives essentially intact,
but its phase is nudged a little every period by the other cell's influence.
The interaction function $H$ summarizes that nudge, and the examples below plot
it, use it to predict phase-locked states, and check the prediction against
direct simulation for identical and non-identical (heterogeneous) pairs.

For phases $\theta_1,\theta_2$, the weak-coupling reduction has the form

$$
\dot\theta_i=\omega_i+\varepsilon H(\theta_j-\theta_i).
$$

Writing the phase difference as $\psi=\theta_2-\theta_1$ and subtracting the two
equations gives a single difference equation $\dot\psi=(\omega_2-\omega_1)+\varepsilon D(\psi)$,
with $D(\psi)=H(-\psi)-H(\psi)$. A zero of $D$ (shifted by any frequency
mismatch) is a locked phase difference; whether the flow around it points back
in on both sides determines if that lock is stable or unstable.

See [`chapter28.md`](chapter28.md) for the full guide, including suggested
order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

## Difference Function With Two Fixed Points

`D_two_fixed_points` is the phase-difference interaction function
$D(\psi)=g_0(\psi)-g_0(1-\psi)$ built from a single-pulse-like PRC $g_0$.
`find_two_fixed_points` locates the two zeros of $D(\psi)-c$ above $\psi=0.5$ by
bisection: the lower one, `psi_unstable`, is an unstable fixed point (flow
moves away from it) and the upper one, `psi_stable`, is stable (flow moves
toward it).

In [ ]:
def g_0(phi):
    phi_tilde = np.mod(phi, 1)
    return phi_tilde ** 2 * (1 - phi_tilde)


def D_two_fixed_points(psi):
    return g_0(psi) - g_0(1 - psi)


def find_two_fixed_points(c=0.08, n_grid=301):
    psi = np.arange(n_grid) / (n_grid - 1)

    psi_0 = 0.5
    for i in range(1, 101):
        psi_i = 0.5 + 0.5 * i / 100
        if D_two_fixed_points(psi_i) > c:
            psi_0 = psi_i

    psi_left, psi_right = 0.5, psi_0
    while psi_right - psi_left > 1e-12:
        psi_c = (psi_left + psi_right) / 2
        if D_two_fixed_points(psi_c) > c:
            psi_right = psi_c
        else:
            psi_left = psi_c
    psi_unstable = (psi_left + psi_right) / 2

    psi_left, psi_right = psi_0, 1.
    while psi_right - psi_left > 1e-12:
        psi_c = (psi_left + psi_right) / 2
        if D_two_fixed_points(psi_c) > c:
            psi_left = psi_c
        else:
            psi_right = psi_c
    psi_stable = (psi_left + psi_right) / 2

    return psi, D_two_fixed_points(psi), c, psi_0, psi_unstable, psi_stable


def plot_d_two_fixed_points(psi, D_vals, c, psi_unstable, psi_stable):
    plt.figure(figsize=(6, 6))
    plt.plot(psi, D_vals, '-k', linewidth=5)
    plt.axis([0, 1, -0.1, 0.1])
    plt.gca().set_box_aspect(1)
    plt.xlabel(r'$\psi$')
    plt.ylabel(r'$D(\psi)$')

    plt.plot([0, 1], [c, c], '--r', linewidth=3)
    plt.text(-0.075, 0.08, '$c$', color='red', fontsize=20)

    plt.plot([psi_unstable, psi_unstable], [-0.1, D_two_fixed_points(psi_unstable)], '--r', linewidth=2)
    plt.plot(psi_unstable, -0.1, 'or', markersize=12, markerfacecolor='w')

    plt.plot([psi_stable, psi_stable], [-0.1, D_two_fixed_points(psi_stable)], '--r', linewidth=2)
    plt.plot(psi_stable, -0.1, 'or', markersize=12, markerfacecolor='r')

    plt.tight_layout()
    plt.show()

In [ ]:
psi, D_vals, c, psi_0, psi_unstable, psi_stable = find_two_fixed_points()
plot_d_two_fixed_points(psi, D_vals, c, psi_unstable, psi_stable)

In [ ]:
def _plot_two_fixed_points_for_c(c=0.08):
    psi, D_vals, c, psi_0, psi_unstable, psi_stable = find_two_fixed_points(c=c)
    plot_d_two_fixed_points(psi, D_vals, c, psi_unstable, psi_stable)


interact(_plot_two_fixed_points_for_c, c=(0.01, 0.09, 0.005));

## Identical Weakly Coupled Pair (Interaction Function 1)

Two ways to track the phase difference of an identical weakly coupled pair:
`simulate_weakly_coupled_event_driven` advances each oscillator exactly to its
next spike (`ceiling` handles the "just spiked" edge case) and applies the
weak-coupling kick from `g` at each spike; `simulate_weakly_coupled_de`
integrates the reduced difference equation $\dot\psi=\varepsilon(g(\psi)-g(-\psi))$
with Heun's method. Both are reused as-is in the next example with a different
`g`; here `wc1_g` gives $g(\varphi)=\varphi^2(1-\varphi)$, and
`simulate_weakly_coupled_1` runs both methods together for one `epsilon`.

In [ ]:
def ceiling(x):
    '''Like ceil, but rounds an exact integer up to the next integer
    instead of leaving it unchanged (phi == an integer means "just
    spiked", not "about to spike").'''
    c = np.ceil(x)
    if c == x:
        c += 1
    return c


def simulate_weakly_coupled_event_driven(g, epsilon, phi_B_0, t_final=None):
    if t_final is None:
        t_final = 6 / epsilon

    phi_A, phi_B = 0., phi_B_0
    t_vec = [0.]
    psi_vec = [phi_B - phi_A]

    t = 0.
    while t < t_final:
        delta_A = ceiling(phi_A) - phi_A
        delta_B = ceiling(phi_B) - phi_B
        if delta_B < delta_A:
            t = t + delta_B
            phi_A = phi_A + delta_B
            phi_A = phi_A + epsilon * g(phi_A)
            phi_B = ceiling(phi_B)
        else:
            t = t + delta_A
            phi_B = phi_B + delta_A
            phi_B = phi_B + epsilon * g(phi_B)
            phi_A = ceiling(phi_A)
        t_vec.append(t)
        psi_vec.append(phi_B - phi_A)

    return np.array(t_vec), np.array(psi_vec)


def simulate_weakly_coupled_de(g, epsilon, phi_B_0, dt=0.01, t_final=None):
    if t_final is None:
        t_final = 6 / epsilon
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    psi = np.zeros(m_steps + 1)
    psi[0] = phi_B_0
    for k in range(m_steps):
        psi_inc = epsilon * (g(psi[k]) - g(-psi[k]))
        psi_tmp = psi[k] + dt05 * psi_inc
        psi_inc = epsilon * (g(psi_tmp) - g(-psi_tmp))
        psi[k + 1] = psi[k] + dt * psi_inc
    t = np.arange(m_steps + 1) * dt
    return t, psi


def wc1_g(phi):
    p = np.mod(phi, 1)
    return p ** 2 * (1 - p)


def simulate_weakly_coupled_1(epsilon, phi_B_0=0.4, dt=0.01):
    t_vec, psi_vec = simulate_weakly_coupled_event_driven(wc1_g, epsilon, phi_B_0)
    t_de, psi_de = simulate_weakly_coupled_de(wc1_g, epsilon, phi_B_0, dt=dt)
    return t_vec, psi_vec, t_de, psi_de


def plot_weakly_coupled(t_vec_1, psi_vec_1, t_de_1, psi_de_1, epsilon_1,
                         t_vec_2, psi_vec_2, t_de_2, psi_de_2, epsilon_2):
    fig, axes = plt.subplots(2, 1, figsize=(8, 8))

    for n in range(len(t_vec_1) - 1):
        axes[0].plot([t_vec_1[n], t_vec_1[n + 1]], [psi_vec_1[n], psi_vec_1[n]], '-k', linewidth=2)
    axes[0].plot(t_de_1, psi_de_1, '-r', linewidth=2)
    axes[0].set_ylabel(r'$\psi$')
    axes[0].set_title(rf'$\epsilon={epsilon_1}$')
    M = np.ceil(max(psi_vec_1.max(), psi_de_1.max()))
    m = np.floor(min(psi_vec_1.min(), psi_de_1.min()))
    axes[0].axis([0, 6 / epsilon_1, m, M])

    for n in range(len(t_vec_2) - 1):
        axes[1].plot([t_vec_2[n], t_vec_2[n + 1]], [psi_vec_2[n], psi_vec_2[n]], '-k', linewidth=2)
    axes[1].plot(t_de_2, psi_de_2, '-r', linewidth=2)
    axes[1].set_ylabel(r'$\psi$')
    axes[1].set_xlabel('$t$ [units of $T$]')
    axes[1].set_title(rf'$\epsilon={epsilon_2}$')
    M = np.ceil(max(psi_vec_2.max(), psi_de_2.max()))
    m = np.floor(min(psi_vec_2.min(), psi_de_2.min()))
    axes[1].axis([0, 6 / epsilon_2, m, M])

    plt.tight_layout()
    plt.show()

In [ ]:
t_vec_1, psi_vec_1, t_de_1, psi_de_1 = simulate_weakly_coupled_1(epsilon=0.5)
t_vec_2, psi_vec_2, t_de_2, psi_de_2 = simulate_weakly_coupled_1(epsilon=0.1)
plot_weakly_coupled(t_vec_1, psi_vec_1, t_de_1, psi_de_1, 0.5,
                     t_vec_2, psi_vec_2, t_de_2, psi_de_2, 0.1)

In [ ]:
def _plot_weakly_coupled_1(epsilon_1=0.5, epsilon_2=0.1):
    plot_weakly_coupled(*simulate_weakly_coupled_1(epsilon=epsilon_1), epsilon_1,
                         *simulate_weakly_coupled_1(epsilon=epsilon_2), epsilon_2)


interact(_plot_weakly_coupled_1, epsilon_1=(0.05, 1.0, 0.05), epsilon_2=(0.05, 1.0, 0.05));

## Identical Weakly Coupled Pair (Interaction Function 2)

Same event-driven/reduced-equation comparison as above, now with a different
interaction function, `wc2_g` with $g(\varphi)=\varphi(1-\varphi)^3$, and a
different initial offset (`phi_B_0=0.1`). `simulate_weakly_coupled_2` reuses
`simulate_weakly_coupled_event_driven` and `simulate_weakly_coupled_de`
unchanged -- only the shape of `g` and the initial condition differ from
`simulate_weakly_coupled_1`.

In [ ]:
def wc2_g(phi):
    p = np.mod(phi, 1)
    return p * (1 - p) ** 3


def simulate_weakly_coupled_2(epsilon, phi_B_0=0.1, dt=0.01):
    t_vec, psi_vec = simulate_weakly_coupled_event_driven(wc2_g, epsilon, phi_B_0)
    t_de, psi_de = simulate_weakly_coupled_de(wc2_g, epsilon, phi_B_0, dt=dt)
    return t_vec, psi_vec, t_de, psi_de

In [ ]:
t_vec_1, psi_vec_1, t_de_1, psi_de_1 = simulate_weakly_coupled_2(epsilon=0.5)
t_vec_2, psi_vec_2, t_de_2, psi_de_2 = simulate_weakly_coupled_2(epsilon=0.1)
plot_weakly_coupled(t_vec_1, psi_vec_1, t_de_1, psi_de_1, 0.5,
                     t_vec_2, psi_vec_2, t_de_2, psi_de_2, 0.1)

In [ ]:
def _plot_weakly_coupled_2(epsilon_1=0.5, epsilon_2=0.1):
    plot_weakly_coupled(*simulate_weakly_coupled_2(epsilon=epsilon_1), epsilon_1,
                         *simulate_weakly_coupled_2(epsilon=epsilon_2), epsilon_2)


interact(_plot_weakly_coupled_2, epsilon_1=(0.05, 1.0, 0.05), epsilon_2=(0.05, 1.0, 0.05));

## Heterogeneous Weakly Coupled Pair

Now oscillator A keeps intrinsic period 1 while B's period is detuned to
$T_B=1+\varepsilon c$: `simulate_heterogeneous_event_driven` advances each
cell to its next spike using this mismatched period, and
`simulate_heterogeneous_de` integrates the drift-added reduced equation
$\dot\psi=\varepsilon\big(g(\psi)-g(-\psi)\big)-c\varepsilon$.
`simulate_weakly_coupled_heterogeneous_1` runs both for a given detuning `c`;
comparing `c=0.08` and `c=0.12` shows how enough mismatch can turn a locked
phase difference into steady drift.

In [ ]:
def het_g(phi):
    p = np.mod(phi, 1)
    return p ** 2 * (1 - p)


def simulate_heterogeneous_de(c, epsilon=0.1, phi_B_0=0.5, dt=0.1, t_final=500.):
    dt05 = dt / 2
    m_steps = round(t_final / dt)
    psi = np.zeros(m_steps + 1)
    psi[0] = phi_B_0
    for k in range(m_steps):
        psi_inc = epsilon * (het_g(psi[k]) - het_g(-psi[k])) - c * epsilon
        psi_tmp = psi[k] + dt05 * psi_inc
        psi_inc = epsilon * (het_g(psi_tmp) - het_g(-psi_tmp)) - c * epsilon
        psi[k + 1] = psi[k] + dt * psi_inc
    t = np.arange(m_steps + 1) * dt
    return t, psi


def simulate_heterogeneous_event_driven(c, epsilon=0.1, phi_B_0=0.5, t_final=500.):
    '''A has intrinsic period 1 (measured in units of T_A); B's period is
    T_B = 1 + epsilon*c, i.e. slightly detuned from A's.'''
    T_B = 1 + epsilon * c
    phi_A, phi_B = 0., phi_B_0
    t_vec = [0.]
    psi_vec = [phi_B - phi_A]

    t = 0.
    while t <= t_final:
        delta_B = (ceiling(phi_B) - phi_B) * T_B
        delta_A = ceiling(phi_A) - phi_A
        if delta_B < delta_A:
            t = t + delta_B
            phi_A = phi_A + delta_B
            phi_A = phi_A + epsilon * het_g(phi_A)
            phi_B = ceiling(phi_B)
        else:
            t = t + delta_A
            phi_B = phi_B + delta_A / T_B
            phi_B = phi_B + epsilon * het_g(phi_B)
            phi_A = ceiling(phi_A)
        t_vec.append(t)
        psi_vec.append(phi_B - phi_A)

    return np.array(t_vec), np.array(psi_vec)


def simulate_weakly_coupled_heterogeneous_1(c, epsilon=0.1, phi_B_0=0.5, dt=0.1, t_final=500.):
    t_de, psi_de = simulate_heterogeneous_de(c, epsilon=epsilon, phi_B_0=phi_B_0, dt=dt, t_final=t_final)
    t_vec, psi_vec = simulate_heterogeneous_event_driven(c, epsilon=epsilon, phi_B_0=phi_B_0, t_final=t_final)
    return t_de, psi_de, t_vec, psi_vec


def plot_weakly_coupled_heterogeneous(t_de_1, psi_de_1, t_vec_1, psi_vec_1, c_1,
                                       t_de_2, psi_de_2, t_vec_2, psi_vec_2, c_2,
                                       epsilon=0.1, t_final=500.):
    fig, axes = plt.subplots(2, 1, figsize=(8, 8))

    axes[0].plot(t_de_1, psi_de_1, '-r', linewidth=2)
    for n in range(len(t_vec_1) - 1):
        axes[0].plot([t_vec_1[n], t_vec_1[n + 1]], [psi_vec_1[n], psi_vec_1[n]], '-k', linewidth=2)
    axes[0].set_ylabel(r'$\psi$')
    axes[0].set_title(rf'$\epsilon={epsilon}$,  $c={c_1}$')
    M = np.ceil(max(psi_vec_1.max(), psi_de_1.max()))
    m = np.floor(min(psi_vec_1.min(), psi_de_1.min()))
    axes[0].axis([0, t_final, m, M])

    axes[1].plot(t_de_2, psi_de_2, '-r', linewidth=2)
    for n in range(len(t_vec_2) - 1):
        axes[1].plot([t_vec_2[n], t_vec_2[n + 1]], [psi_vec_2[n], psi_vec_2[n]], '-k', linewidth=2)
    axes[1].set_ylabel(r'$\psi$')
    axes[1].set_xlabel('$t$ [units of $T_A$]')
    axes[1].set_title(rf'$\epsilon={epsilon}$,  $c={c_2}$')
    M = np.ceil(max(psi_vec_2.max(), psi_de_2.max()))
    m = np.floor(min(psi_vec_2.min(), psi_de_2.min()))
    axes[1].axis([0, t_final, m, M])

    plt.tight_layout()
    plt.show()

In [ ]:
t_de_1, psi_de_1, t_vec_1, psi_vec_1 = simulate_weakly_coupled_heterogeneous_1(c=0.08)
t_de_2, psi_de_2, t_vec_2, psi_vec_2 = simulate_weakly_coupled_heterogeneous_1(c=0.12)
plot_weakly_coupled_heterogeneous(t_de_1, psi_de_1, t_vec_1, psi_vec_1, 0.08,
                                   t_de_2, psi_de_2, t_vec_2, psi_vec_2, 0.12)

In [ ]:
def _plot_weakly_coupled_heterogeneous(c_1=0.08, c_2=0.12):
    plot_weakly_coupled_heterogeneous(*simulate_weakly_coupled_heterogeneous_1(c=c_1), c_1,
                                       *simulate_weakly_coupled_heterogeneous_1(c=c_2), c_2)


interact(_plot_weakly_coupled_heterogeneous, c_1=(0.0, 0.2, 0.01), c_2=(0.0, 0.2, 0.01));